# T2.4 - DBRepo Views

This notebook creates or records the DBRepo views for the frost-day experiment.

The full SQL versions are in `docs/views.sql`. The DBRepo client used here can create simple column-selection views, but not every SQL feature used in the final view definitions. Cases that cannot be created cleanly are recorded instead of creating a wrong view with the right name.

Known IDs:
- Database: `a16a980a-3489-492b-adcf-74c70a07248b`
- Table `daily_observation`: `21e39deb-1db9-4d34-bc04-45fa5cef72a0`
- Table `station`: `9c57f7ff-99b6-454b-8d51-cff340ecf934`


## 1 - Dependency

If the DBRepo client is missing:

```bash
python -m pip install dbrepo
```


In [ ]:
import os
from pathlib import Path

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition


def load_env_file(path=".env"):
    for env_path in (Path.cwd() / path, Path.cwd().parent / path):
        if not env_path.exists():
            continue
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())
        return


load_env_file()

client = RestClient(
    endpoint=os.getenv("DBREPO_ENDPOINT", "https://test.dbrepo.tuwien.ac.at"),
    username=os.getenv("DBREPO_USERNAME"),
    password=os.getenv("DBREPO_PASSWORD"),
)

DATABASE_ID = os.getenv("DATABASE_ID")
OBS_TABLE_ID = os.getenv("OBS_TABLE_ID")
STATION_TABLE_ID = os.getenv("STATION_TABLE_ID")

if not all([DATABASE_ID, OBS_TABLE_ID, STATION_TABLE_ID]):
    raise ValueError("DATABASE_ID, OBS_TABLE_ID, and STATION_TABLE_ID must be set in .env")

print("Logged in as:", client.whoami())


In [ ]:
import inspect

print(inspect.signature(client.create_view))


In [ ]:
views = client.get_views(database_id=DATABASE_ID)
print(f"Existing views: {len(views)}")
for view in views:
    print(f"{view.name}: {view.id}")


In [ ]:
obs_table = client.get_table(database_id=DATABASE_ID, table_id=OBS_TABLE_ID)
station_table = client.get_table(database_id=DATABASE_ID, table_id=STATION_TABLE_ID)

print("daily_observation")
for col in obs_table.columns:
    print(f"- {col.name}")

print()
print("station")
for col in station_table.columns:
    print(f"- {col.name}")


In [ ]:
print("QueryDefinition fields:")
for name in QueryDefinition.model_fields:
    print(f"- {name}")


In [ ]:
def create_simple_observation_view(view_name):
    existing_names = [view.name for view in client.get_views(database_id=DATABASE_ID)]
    if view_name in existing_names:
        print(f"View already exists: {view_name}")
        return

    query = QueryDefinition(
        datasources=["daily_observation"],
        columns=[
            "daily_observation.obs_date",
            "daily_observation.temp_mean_c",
            "daily_observation.temp_max_c",
            "daily_observation.temp_min_c",
            "daily_observation.precipitation_mm",
            "daily_observation.sunshine_h",
            "daily_observation.humidity_pct",
            "daily_observation.visibility_m",
        ],
    )
    result = client.create_view(
        database_id=DATABASE_ID,
        name=view_name,
        query=query,
        is_public=True,
        is_schema_public=True,
    )
    print(f"Created {view_name}: {result.id}")

create_simple_observation_view("vw_ml_daily_features")


In [ ]:
create_simple_observation_view("vw_ml_complete_cases")


In [ ]:
view_name = "vw_ml_balanced_training_sample"
existing = next((view for view in client.get_views(database_id=DATABASE_ID) if view.name == view_name), None)

if existing:
    print(f"{view_name}: {existing.id}")
    print("Check manually that this is really the balanced SQL view from docs/views.sql.")
else:
    print(f"{view_name}: not created by the DBRepo client")
    print("The SQL definition is in docs/views.sql.")


In [ ]:
view_name = "vw_monthly_frost_summary"
existing = next((view for view in client.get_views(database_id=DATABASE_ID) if view.name == view_name), None)

if existing:
    print(f"{view_name}: {existing.id}")
    print("Check manually that this is really the monthly aggregate SQL view from docs/views.sql.")
else:
    print(f"{view_name}: not created by the DBRepo client")
    print("The SQL definition is in docs/views.sql.")


In [ ]:
import requests

wanted = {
    "vw_ml_daily_features",
    "vw_ml_complete_cases",
    "vw_ml_balanced_training_sample",
    "vw_monthly_frost_summary",
}

base_url = os.getenv("DBREPO_ENDPOINT", "https://test.dbrepo.tuwien.ac.at").rstrip("/")
auth = (os.getenv("DBREPO_USERNAME"), os.getenv("DBREPO_PASSWORD"))

for view in client.get_views(database_id=DATABASE_ID):
    if view.name not in wanted:
        continue

    url = f"{base_url}/api/v1/database/{DATABASE_ID}/view/{view.id}/data"
    try:
        response = requests.get(
            url,
            params={"page": 0, "size": 5},
            auth=auth,
            headers={"Accept": "application/json"},
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        rows = data if isinstance(data, list) else data.get("data", data)
        row_count = len(rows) if isinstance(rows, list) else "response received"
        print(f"{view.name}: preview ok ({row_count})")
    except Exception as exc:
        print(f"{view.name}: preview failed: {exc}")


In [ ]:
wanted = [
    "vw_ml_daily_features",
    "vw_ml_complete_cases",
    "vw_ml_balanced_training_sample",
    "vw_monthly_frost_summary",
]

views_by_name = {
    view.name: view
    for view in client.get_views(database_id=DATABASE_ID)
    if view.name in wanted
}

print("Relevant view IDs")
for name in wanted:
    view = views_by_name.get(name)
    if view:
        print(f"{name}: {view.id}")
    else:
        print(f"{name}: not present in DBRepo")

print()
print("Feature view for .env")
view = views_by_name.get("vw_ml_complete_cases")
if view:
    print(f"DBREPO_FEATURE_VIEW_ID={view.id}")
else:
    print("DBREPO_FEATURE_VIEW_ID=<missing>")
